# Data aggregation for 2026 matches

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

Load data

In [ ]:
model_choice = input(
    "Choose model:\n"
    "* logreg  - Logistic Regression\n"
    "* rf      - Random Forest Classifier\n"
    "* extratrees      - ExtraTrees Classifier\n"
    "* gb      - Gradient Boosting Classifier\n"
    "* histgb     - Histogram Gradient Boosting Classifier\n"
).strip().lower()

predictions = pd.read_csv(f"predictions/{model_choice}/predictions_2026_group_matches.csv")
predictions.head()

knockout_matches = pd.read_csv("matches/2026_knockout_matches.csv")

Analyse data

In [ ]:
predictions = predictions.sort_values(
    by=["group", "confidence"],
    ascending=[True, False]
)
wins = predictions[["group", "predicted_winner"]].copy()

wins["wins"] = 1

group_wins = (
    wins
    .groupby(["group", "predicted_winner"])
    .sum()
    .reset_index()
    .rename(columns={"predicted_winner": "team"})
)

group_wins["rank"] = (
    group_wins
    .groupby("group")["wins"]
    .rank(method="first", ascending=False)
).astype(int)

group_wins = group_wins.sort_values(
    by=["group", "rank"],
    ascending=[True, True]
)

group_wins = group_wins[group_wins["rank"] <= 3]

Output group win predictions as CSV

In [ ]:
print(group_wins)
group_wins.to_csv(
    f"predictions/{model_choice}/predictions_2026_group_wins.csv",
    index=False
)

Fill knockout matches with group winners

In [ ]:
winners = (
    group_wins[group_wins["rank"] == 1]
    .set_index("group")["team"]
    .to_dict()
)

runners_up = (
    group_wins[group_wins["rank"] == 2]
    .set_index("group")["team"]
    .to_dict()
)

third_place = group_wins[group_wins["rank"] == 3][["group", "team"]]

team_confidence = pd.concat([
    predictions[["group", "team1", "confidence"]]
        .rename(columns={"team1": "team"}),

    predictions[["group", "team2", "confidence"]]
        .rename(columns={"team2": "team"})
])

third_place_strength = (
    third_place
    .merge(team_confidence, on=["group", "team"])
    .groupby(["group", "team"])["confidence"]
    .mean()
    .reset_index(name="avg_confidence")
    .sort_values(by="avg_confidence", ascending=False)
)

third_place_strength.head(8)

In [ ]:
def resolve_team(value):
    if value.startswith("WINNER_"):
        group = value.split("_")[1]
        return winners.get(group, value)

    if value.startswith("RUNNER_UP_"):
        group = value.split("_")[2]
        return runners_up.get(group, value)

    if value.startswith("THIRD_"):
        idx = int(value.split("_")[1]) - 1
        return third_place_strength.loc[idx, "team"]

    return value


knockout_matches.loc[:, "team1"] = knockout_matches["team1"].apply(resolve_team)
knockout_matches.loc[:, "team2"] = knockout_matches["team2"].apply(resolve_team)

Output team predictions of knockout matches

In [ ]:
knockout_matches.to_csv(
    f"predictions/{model_choice}/predictions_2026_knockout_matches.csv",
    index=False
)

knockout_matches.head(16)